## ClavaDDPM Training

In [10]:
import os
from pathlib import Path


# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/home/coder/vector-bootcamp-synthetic-data')

In [11]:
import pickle
from logging import INFO

from hydra import compose, initialize
from midst_toolkit.common.config import ClavaDDPMClassifierConfig, ClavaDDPMClusteringConfig, ClavaDDPMDiffusionConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.common.random import set_all_random_seeds
from midst_toolkit.common.variables import DEVICE
from midst_toolkit.models.clavaddpm.clustering import clava_clustering
from midst_toolkit.models.clavaddpm.data_loaders import Table, load_tables
from midst_toolkit.models.clavaddpm.train import ClavaDDPMModelArtifacts, clava_training
from omegaconf import OmegaConf


# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)

## Set the paths and load the hydra config

In [12]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
print("IMPLEMENTATION ROOT: ", IMPLEMENTATION_ROOT)
# Set data and output directories
# Berka dataset: https://webpages.charlotte.edu/mirsad/itcs6265/group1/domain.html
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"

IMPLEMENTATION ROOT:  /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data


In [13]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))

# Load and set the random seeds
if cfg.random_seed is not None:
    set_all_random_seeds(seed=cfg.random_seed)

INFO :      Setting random seeds to 42.


diffusion_config:
  d_layers:
  - 512
  - 1024
  - 1024
  - 1024
  - 1024
  - 512
  dropout: 0.0
  num_timesteps: 500
  model_type: mlp
  iterations: 5000
  batch_size: 256
  lr: 0.0006
  gaussian_loss_type: mse
  weight_decay: 1.0e-05
  scheduler: cosine
  data_split_ratios:
  - 0.99
  - 0.005
  - 0.005
  merge_categoricals_into_numerical: true
random_seed: 42
clustering_config:
  parent_scale: 1.0
  num_clusters: 50
  clustering_method: kmeans_and_gmm
classifier_config:
  d_layers:
  - 128
  - 256
  - 512
  - 1024
  - 512
  - 256
  - 128
  lr: 0.0001
  dim_t: 128
  batch_size: 256
  iterations: 5000



### Step 1: Load the tables

In [14]:
# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)
# `load_tables` loads tables f"{table}.csv" saved under `base_data_dir` and
# their domain files as f"{table}_domain.json".
log(INFO, f"Loading data from {base_data_dir}...")
tables, relation_order, _ = load_tables(Path(base_data_dir))

INFO :      Loading data from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/data/berka...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (77, 15)
INFO :      Total dataframe shape: (77, 15)
INFO :      Numerical data shape: (77, 13)
INFO :      Categorical data shape: (77, 2)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (5369, 4)
INFO :      Total dataframe shape: (5369, 4)
INFO :      Numerical data shape: (5369, 0)
INFO :      Categorical data shape: (5369, 4)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (4500, 2)
INFO :      Total dataframe shape: (4500, 2)
INFO :      Numerical data shape: (4500, 1)
INFO :      Categorical data shape: (4500, 1)
INFO :      Training data ratio is 1, so the d

## Step 2: Cluster the data
If there is a clustering checkpoint saved at `base_output_dir/clustering_ckpt.pkl`, the `clava_clustering` will load and use it. Remember to remove old clustering checkpoints.

In [16]:
log(INFO, "Clustering step...")
clustering_config = ClavaDDPMClusteringConfig(**cfg.clustering_config)
tables, _ = clava_clustering(tables, relation_order, Path(base_output_dir), clustering_config)
log(INFO, "Clustering checkpoint is saved at `base_output_dir/clustering_ckpt.pkl`.")

INFO :      Clustering step...
INFO :      Clustering account -> trans


Initialization 0
  Iteration 10
  Iteration 20
  Iteration 30
  Iteration 40
  Iteration 50
  Iteration 60
Initialization converged.


INFO :      Average agree rate: 0.32461162779111696
INFO :      Number of cluster centers: 48
INFO :      Clustering account -> order


Initialization 0
  Iteration 10
  Iteration 20


INFO :      Average agree rate: 0.8110031931878658
INFO :      Number of cluster centers: 47
INFO :      Clustering account -> loan


...

INFO :      Average agree rate: 0.404865760888154
INFO :      Number of cluster centers: 35
INFO :      Clustering district -> client


  Iteration 10
Initialization converged.
Initialization 0


INFO :      Average agree rate: 0.7634681668933135
INFO :      Number of cluster centers: 48


  Iteration 10
Initialization converged.


INFO :      Clustering checkpoint is saved at `base_output_dir/clustering_ckpt.pkl`.


## Step 3: Train the model

In [18]:
log(INFO, "Training model...")
diffusion_config = ClavaDDPMDiffusionConfig(**cfg.diffusion_config)
classifier_config = ClavaDDPMClassifierConfig(**cfg.classifier_config)

tables, _ = clava_training(
    tables,
    relation_order,
    Path(base_output_dir),
    diffusion_config,
    classifier_config,
    device=DEVICE,
)
log(INFO, "Model trained successfully.")
# Note: It takes about 20 minutes to run on local with default values,
# depending on the hardware.

INFO :      Training model...
INFO :      Training None -> district model from scratch
INFO :      No cache_dir provided. Will not attempt to load or save transformed dataset from/to cache
INFO :      No NaN processing policy specified.
INFO :      Model params: ModelParameters(diffusion_parameters=DiffusionParameters(layers_dimensions=[512, 1024, 1024, 1024, 1024, 512], dropout=0.0, input_dimension=0, output_dimension=0, embedding_dimension=0, n_blocks=0, block_dimension=0, hidden_dimension=0, dropout_first=0, dropout_second=0), input_dimension=np.int64(17), num_classes=0, is_target_conditioned=<IsTargetConditioned.NONE: 'none'>)
INFO :      Getting model: mlp
INFO :      Step 500/5000 MLoss: 0.0 GLoss: 0.3305 Sum: 0.3305
INFO :      Step 1000/5000 MLoss: 0.0 GLoss: 0.2441 Sum: 0.2441
INFO :      Step 1500/5000 MLoss: 0.0 GLoss: 0.2186 Sum: 0.2186
INFO :      Step 2000/5000 MLoss: 0.0 GLoss: 0.1962 Sum: 0.1962
INFO :      Step 2500/5000 MLoss: 0.0 GLoss: 0.1852 Sum: 0.1852
INFO :     

## Step 4: Load  the cluster checkpoint and the trained models

In [19]:
clustering_results_file = Path(base_output_dir) / "cluster_ckpt.pkl"
with open(clustering_results_file, "rb") as f:
    clustering_result = pickle.load(f)

assert all(isinstance(table, Table) for table in clustering_result["tables"].values())
assert isinstance(clustering_result["all_group_lengths_prob_dicts"], dict)
log(INFO, "Clustering checkpoint is loaded successfully.")

INFO :      Clustering checkpoint is loaded successfully.


In [20]:
for relation in relation_order:
    results_file = Path(base_output_dir) / "models" / f"{relation[0]}_{relation[1]}_ckpt.pkl"
    log(INFO, f"Checking the results from {results_file}...")

    with open(results_file, "rb") as f:
        result = pickle.load(f)

    # Asserting the results are the correct type
    assert isinstance(result, ClavaDDPMModelArtifacts)

    log(INFO, f"Result size (in bytes): {results_file.stat().st_size}")

INFO :      Checking the results from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/models/None_district_ckpt.pkl...
INFO :      Result size (in bytes): 18453730
INFO :      Checking the results from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/models/district_client_ckpt.pkl...
INFO :      Result size (in bytes): 42709468
INFO :      Checking the results from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/models/district_account_ckpt.pkl...
INFO :      Result size (in bytes): 42720969
INFO :      Checking the results from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/models/client_disp_ckpt.pkl...
INFO :      Result size (in bytes): 42608541
INFO :      Checking the results from /home/coder/vector-bootcamp-synthetic-data/implementations/tabular_data/multi_table/results/models/account_disp_ckpt.pkl...
INF